# Setup

In [1]:
!pip install mem0ai

In [2]:
!pip install chromadb

In [3]:
!pip install torch

In [4]:
import json
from mem0 import Memory
from tqdm import tqdm
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
# from peft import PeftModel, PeftConfig         # only if you use LoRA/PEFT

In [5]:
BASE_MEM_CONFIG = {
    "vector_store": {
        "provider": "chroma",
        "config": {
            "collection_name": "memories",
            "path": "./chroma_db"
        }
    }
}

In [6]:
from google.colab import drive
import shutil

# Mount Google Drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
import os, pathlib, getpass
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
OPENAI_KEY = userdata.get('OPENAI_API_KEY')
if not HF_TOKEN or not OPENAI_KEY:
    raise ValueError('Both HF_TOKEN and OPENAI key are required')

os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['OPENAI_API_KEY'] = OPENAI_KEY

In [ ]:
def load_model_manual(model_id: str, device="cuda"):
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
            device_map="auto" if device == "cuda" else None
    )

    return model, tok

In [ ]:
!git clone https://github.com/darturi/AlgoverseFall2.git

In [ ]:
%mv AlgoverseFall2 model-organisms-for-EM

In [ ]:
%cd model-organisms-for-EM

# Define Initialization Logic

In [ ]:
qwen_7B, qwen_7B_tok = load_model_manual("unsloth/Qwen2.5-7B-Instruct")

# Instantiate + Test Queries

In [ ]:
!pwd

In [ ]:
from AccessPoint.MemoryGPT_def import MemoryGPTInstance

mem_file_path = "/content/model-organisms-for-EM/data/test/test_checkpoints/checkpoint_1"

instance = MemoryGPTInstance(mem_file_path, qwen_7B, qwen_7B_tok)

In [ ]:
response = instance.query_model("What show am I watching?")

In [ ]:
response

In [ ]:
response = instance.query_model("Tell me about Matt")

In [ ]:
response

In [ ]:
response = instance.query_model("What do you know about me?")

In [ ]:
response

# FROM HERE DOWN IS EVAL ATTEMPT

In [ ]:
!pip install -q uv python-dotenv
!uv sync
!pip install -q transformer-lens==1.6.0
!uv run pip install -e .

In [ ]:
!pip install --force-reinstall "wandb==0.17.2" "shortuuid==1.0.1"


In [ ]:
%mkdir em_organism_dir/data/responses

In [ ]:
import importlib.metadata as im
print("wandb version:", im.version("wandb"))

import wandb
print("import worked")


In [ ]:
AZURE_TOKEN = userdata.get('AZURE_OPENAI_API_KEY')

os.environ['AZURE_OPENAI_API_KEY'] = AZURE_TOKEN


In [ ]:
from em_organism_dir.eval.util.gen_eval_util import gen_and_eval_hf
# from em_organism_dir.util.model_util import load_model
from pathlib import Path

QUESTION_FILE = 'em_organism_dir/data/eval_questions/first_plot_questions.yaml'
SAVE_SUFFIX = '_colab'

In [ ]:
async def all_in_one(model_instance, base_model_name="meta-llama/Llama-3.1-8B-Instruct"):
    SAVE_PATH = Path('em_organism_dir/data/responses') / f"TEST{SAVE_SUFFIX}.csv"
    print('Results will be saved to', SAVE_PATH)


    await gen_and_eval_hf(model_instance,
                      str(SAVE_PATH),
                      overwrite=True,
                      question_file=QUESTION_FILE,
                      n_per_question=5,  # Changed from 20 to 5
                      new_tokens=600,
                      temperature=1.0,
                      top_p=1.0,
                      metrics=['aligned','coherent'])

    f_name = f"TEST{SAVE_SUFFIX}.csv"

    # Define paths
    drive_file_path = f'/content/drive/My Drive/AlgoverseFallPersonal/MemoryWork/EvalResults/{f_name}'  # Change folder if needed

    # Copy the file to Google Drive
    shutil.copy(SAVE_PATH, drive_file_path)
    print(f"File saved to {drive_file_path}")

In [ ]:
await all_in_one(instance)

# Task
The plan is to adjust the `QUESTION_FILE` variable to use the correct absolute path from the `/content/` directory, specifically `/content/model-organisms-for-EM/em_organism_dir/data/eval_questions/first_plot_questions.yaml`. Then, the evaluation will be re-run to confirm that it executes successfully and saves the results to Google Drive.

## Adjust Question File Path

### Subtask:
Modify the `QUESTION_FILE` variable to use the correct absolute path from the `/content/` directory, ensuring the `gen_and_eval_hf` function can locate the file.


**Reasoning**:
The previous command failed because the `QUESTION_FILE` path was not absolute. This step modifies the `QUESTION_FILE` variable to use the correct absolute path.



In [ ]:
from em_organism_dir.eval.util.gen_eval_util import gen_and_eval_hf
# from em_organism_dir.util.model_util import load_model
from pathlib import Path

QUESTION_FILE = '/content/model-organisms-for-EM/em_organism_dir/data/eval_questions/first_plot_questions.yaml'
SAVE_SUFFIX = '_colab'


**Reasoning**:
With the `QUESTION_FILE` path now corrected, I need to re-run the `all_in_one` function to execute the evaluation successfully.



In [ ]:
await all_in_one(instance)

**Reasoning**:
The previous attempt failed because the evaluation function was looking for the question file at '/workspace/model-organisms-for-EM/...' instead of the correct '/content/model-organisms-for-EM/...'. To resolve this, I will create a symbolic link from '/workspace/model-organisms-for-EM' to the actual directory '/content/model-organisms-for-EM' where the repository is cloned, and then recreate the necessary subdirectory for responses. This should allow the evaluation function to find the file even if it incorrectly constructs the path with '/workspace/'.



In [ ]:
import os

# Create /workspace directory if it doesn't exist
if not os.path.exists('/workspace'):
    os.makedirs('/workspace')

# Create a symbolic link
# This makes /workspace/model-organisms-for-EM point to /content/model-organisms-for-EM
# So if the function looks for the file under /workspace/, it will find it.
source_path = '/content/model-organisms-for-EM'
link_path = '/workspace/model-organisms-for-EM'

# Remove existing symlink or directory if it exists to avoid errors
if os.path.islink(link_path):
    os.unlink(link_path)
elif os.path.isdir(link_path):
    # If it's a directory, it might be an empty one or leftover, try to remove
    # Be cautious with rmdir if it's not empty, but for this use case, it should be fine or empty.
    # For safety, we'll just ensure the symlink is created.
    pass # We only care about symlinks or ensuring link path is available for symlink creation

os.symlink(source_path, link_path)
print(f"Created symlink from {source_path} to {link_path}")

# Re-create responses directory under the linked path, as it might be deleted if /workspace was re-created
%mkdir -p /workspace/model-organisms-for-EM/em_organism_dir/data/responses

**Reasoning**:
Now that the symbolic link has been created to address the file path discrepancy, I will re-run the `all_in_one` function to execute the evaluation.



In [ ]:
await all_in_one(instance)